# Week 2 Lasso, Ridge & Elastic Net

**Datasets:** Olist Brazilian E-Commerce

## Topics
1. **Regularized regression (Olist)** — predict review_score from order economics when predictors overlap.
2. **Alpha selection** — compare Ridge validation RMSE across a grid of penalty values.


## Data loading and cleaning

Load and prepare modeling dataframes for all three datasets.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_reviews_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_payments_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_customers_dataset.csv")

pay = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "max"),
).reset_index()
itm = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
).reset_index()
rev = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()

olist_df = (
    orders.merge(rev, on="order_id")
    .merge(itm, on="order_id")
    .merge(pay, on="order_id")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
)
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_month"] = olist_df["order_purchase_timestamp"].dt.month
olist_df["delivery_days"] = (
    pd.to_datetime(olist_df["order_delivered_customer_date"], errors="coerce")
    - olist_df["order_purchase_timestamp"]
).dt.days
olist_df["delivery_days"] = olist_df["delivery_days"].fillna(olist_df["delivery_days"].median())
olist_df = olist_df.dropna(subset=["review_score", "payment_value", "total_price"])

ucl_raw = pd.read_excel(DATA_DIR / "ucl_dataset.xlsx")
ucl_raw = ucl_raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"})
ucl_raw["InvoiceDate"] = pd.to_datetime(ucl_raw["InvoiceDate"])
ucl_raw = ucl_raw.dropna(subset=["CustomerID"])
ucl_raw = ucl_raw[~ucl_raw["InvoiceNo"].astype(str).str.startswith("C", na=False)]
ucl_raw = ucl_raw[(ucl_raw["Quantity"] > 0) & (ucl_raw["UnitPrice"] > 0)]
ucl_raw["LineTotal"] = ucl_raw["Quantity"] * ucl_raw["UnitPrice"]
anchor = ucl_raw["InvoiceDate"].max() + pd.Timedelta(days=1)
ucl_df = (
    ucl_raw.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda x: (anchor - x.max()).days),
        frequency=("InvoiceNo", "nunique"),
        monetary=("LineTotal", "sum"),
        avg_basket=("LineTotal", "mean"),
        n_items=("Quantity", "sum"),
    )
    .reset_index()
)
ucl_df["log_monetary"] = np.log1p(ucl_df["monetary"])
ucl_df["log_frequency"] = np.log1p(ucl_df["frequency"])

TAOBAO_PATH = DATA_DIR / "TaobaoUserBehavior.csv"
user_set = set()
chunks = []
for chunk in pd.read_csv(
    TAOBAO_PATH,
    header=None,
    names=["user_id", "item_id", "category_id", "behavior", "timestamp"],
    chunksize=500_000,
):
    chunk = chunk[chunk["user_id"].isin(user_set) | (len(user_set) < 50000)]
    if len(user_set) < 50000:
        for u in chunk["user_id"].unique():
            if len(user_set) >= 50000:
                break
            user_set.add(u)
    chunk = chunk[chunk["user_id"].isin(user_set)]
    if len(chunk):
        chunks.append(chunk)
    if len(user_set) >= 50000 and sum(len(c) for c in chunks) > 200_000:
        break

taobao = pd.concat(chunks, ignore_index=True)
taobao_df = (
    taobao.groupby("user_id")
    .agg(
        n_events=("behavior", "count"),
        n_pv=("behavior", lambda x: (x == "pv").sum()),
        n_cart=("behavior", lambda x: (x == "cart").sum()),
        n_fav=("behavior", lambda x: (x == "fav").sum()),
        n_buy=("behavior", lambda x: (x == "buy").sum()),
        n_categories=("category_id", "nunique"),
        n_items=("item_id", "nunique"),
    )
    .reset_index()
)
taobao_df["cart_rate"] = taobao_df["n_cart"] / taobao_df["n_events"].clip(lower=1)
taobao_df["converted"] = (taobao_df["n_buy"] > 0).astype(int)

print(f"Olist orders: {len(olist_df):,}")
print(f"UCL customers: {len(ucl_df):,}")
print(f"Taobao users: {len(taobao_df):,}")


# Linear regression on Olist review score

Regress review_score on order-level features.


In [ ]:
import statsmodels.api as sm

olist_features = [
    "total_price", "freight_value", "n_items", "payment_installments",
    "n_payments", "order_month", "delivery_days",
]
df = olist_df[olist_features + ["review_score"]].dropna().copy()

X_all = df[olist_features]
y = df["review_score"]
ols_model = sm.OLS(y, sm.add_constant(X_all)).fit()

print(ols_model.summary())


# Ridge alpha grid search

Select penalty by validation RMSE on a held-out test set.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

X = df[olist_features].values
y = df["review_score"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

alphas = [0.01, 0.05, 0.1, 1.0]
best_alpha = {}
for a in alphas:
    ridge = Ridge(alpha=a).fit(X_train_s, y_train)
    rmse = float(np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_s))))
    best_alpha[a] = rmse
    print(f"alpha = {a}: test RMSE = {rmse:.4f}")

chosen_alpha = min(best_alpha, key=best_alpha.get)
print(f"\nChosen alpha: {chosen_alpha}")


# Lasso, Ridge, and Elastic Net comparison


In [ ]:
from sklearn.linear_model import Lasso, ElasticNet

models = {
    "Lasso": Lasso(alpha=chosen_alpha, max_iter=5000),
    "Ridge": Ridge(alpha=chosen_alpha),
    "ElasticNet": ElasticNet(alpha=chosen_alpha, l1_ratio=0.5, max_iter=5000),
}
scores = {}
coefs = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    scores[name] = {
        "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "r2": float(r2_score(y_test, pred)),
    }
    coefs[name] = model.coef_.tolist()
    print(f"{name}: R² = {scores[name]['r2']:.4f}, RMSE = {scores[name]['rmse']:.4f}")


Order economics alone explain a modest share of review-score variation. Ridge keeps all predictors and stabilizes coefficients when total_price and payment features overlap.


# Coefficient comparison plot


In [ ]:
import matplotlib.pyplot as plt

x_pos = np.arange(len(olist_features))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
for i, (name, c) in enumerate(coefs.items()):
    ax.bar(x_pos + i * width, c, width, label=name)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(olist_features, rotation=30, ha="right")
ax.set_ylabel("Coefficient")
ax.set_title("Week 2: Regularized regression coefficients (Olist review score)")
ax.legend()
plt.tight_layout()
plt.show()
